# Equilibrium Climate Sensitivity (ECS) from CMIP6 data hosted on GDEX
- This workflow is an adaptation of https://gallery.pangeo.io/repos/pangeo-gallery/cmip6/ECS_Gregory_method.html
- We use the [Gregory method](https://agupubs.onlinelibrary.wiley.com/doi/epdf/10.1029/2003GL018747) to compute ECS

## Table of Contents
- [Introduction](#Introduction) 
- [Select Dask Cluster](#Dask-Cluster) 
- [Data Loading](#Data-Loading) 
- [Data Analysis](#Data-Analysis) 

## Introduction
- Load python packkages
- Load catalog url

In [3]:
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import dask
from tqdm.autonotebook import tqdm
import intake
import os
import seaborn as sns
import pandas as pd

In [10]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [4]:
# Set up your sratch folder path
username       = os.environ["USER"]
glade_scratch  = "/glade/derecho/scratch/" + username
print(glade_scratch)

/glade/derecho/scratch/harshah


In [5]:
cat_url     = 'https://osdata.gdex.ucar.edu/d010096/catalogs/d010096-https.json'
# cat_url     = 'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'

## Dask Cluster

In [6]:
# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-wk25-mdm',
    cores = 1,
    memory = '16GiB',
    processes = 1,
    local_directory = glade_scratch+'/dask/spill',
    log_directory = glade_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=1:mem=16GB',
    queue = 'casper',
    walltime = '5:00:00',
    interface = 'ext'
)

/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41785 instead
  warnings.warn(


In [11]:
# Scale the cluster and display cluster dashboard URL
n_workers = 6
client = Client(cluster)
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)
cluster

PBSCluster(b85d409a, 'tcp://128.117.208.69:36589', workers=6, threads=6, memory=96.00 GiB)

## Data Loading
- Load catalog and select data subset

In [29]:
col = intake.open_esm_datastore(cat_url)
col.df

,path,variable,format,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,variant_label,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,https://osdata.gdex.ucar.edu/d010096/AS-RCEC/T...,clivi,zarr,CMIP,AS-RCEC,TaiESM1,1pctCO2,<NA>,Amon,clivi,gn,r1i1p1f1,clivi,Ice Water Path,kg m-2,0,1313256,<NA>,<NA>,708
1,https://osdata.gdex.ucar.edu/d010096/AS-RCEC/T...,clt,zarr,CMIP,AS-RCEC,TaiESM1,1pctCO2,<NA>,Amon,clt,gn,r1i1p1f1,clt,Total Cloud Cover Percentage,%,0,1313256,<NA>,<NA>,708
2,https://osdata.gdex.ucar.edu/d010096/AS-RCEC/T...,clwvi,zarr,CMIP,AS-RCEC,TaiESM1,1pctCO2,<NA>,Amon,clwvi,gn,r1i1p1f1,clwvi,Condensed Water Path,kg m-2,0,1313256,<NA>,<NA>,708
3,https://osdata.gdex.ucar.edu/d010096/AS-RCEC/T...,co2mass,zarr,CMIP,AS-RCEC,TaiESM1,1pctCO2,<NA>,Amon,co2mass,gm,r1i1p1f1,co2mass,Total Atmospheric Mass of CO2,kg,0,1313256,<NA>,<NA>,708
4,https://osdata.gdex.ucar.edu/d010096/AS-RCEC/T...,evspsbl,zarr,CMIP,AS-RCEC,TaiESM1,1pctCO2,<NA>,Amon,evspsbl,gn,r1i1p1f1,evspsbl,Evaporation Including Sublimation and Transpir...,kg m-2 s-1,0,1313256,<NA>,<NA>,708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221094,https://osdata.gdex.ucar.edu/d010096/UA/MCM-UA...,vsf,zarr,CMIP,UA,MCM-UA-1-0,piControl,<NA>,Omon,vsf,gn,r1i1p1f1,vsf,Virtual Salt Flux into Sea Water ...,kg m-2 s-1,0,4379256,<NA>,<NA>,708
221095,https://osdata.gdex.ucar.edu/d010096/UA/MCM-UA...,wo,zarr,CMIP,UA,MCM-UA-1-0,piControl,<NA>,Omon,wo,gn,r1i1p1f1,wo,Sea Water Vertical Velocity ...,m s-1,0,4379256,<NA>,<NA>,708
221096,https://osdata.gdex.ucar.edu/d010096/UA/MCM-UA...,sithick,zarr,CMIP,UA,MCM-UA-1-0,piControl,<NA>,SImon,sithick,gn,r1i1p1f1,sithick,Sea Ice Thickness ...,m,0,4379256,<NA>,<NA>,708
221097,https://osdata.gdex.ucar.edu/d010096/UA/MCM-UA...,areacella,zarr,CMIP,UA,MCM-UA-1-0,piControl,<NA>,fx,areacella,gn,r1i1p1f1,areacella,Grid-Cell Area for Atmospheric Grid Variables,m2,<NA>,<NA>,<NA>,<NA>,<NA>


In [28]:
[eid for eid in col.df['experiment_id'].unique() if 'ssp' in eid]

[]

In [14]:
query = dict(
    experiment_id=['abrupt-4xCO2','piControl'], # pick the `abrupt-4xCO2` and `piControl` forcing experiments
    table_id='Amon',                            # choose to look at atmospheric variables (A) saved at monthly resolution (mon)
    variable_id=['tas', 'rsut','rsdt','rlut'],  # choose to look at near-surface air temperature (tas) as our variable
    member_id = 'r1i1p1f1',                     # arbitrarily pick one realization for each model (i.e. just one set of initial conditions)
)

col_subset = col.search(require_all_on=["source_id"], **query)
col_subset.df.groupby("source_id")[
    ["experiment_id", "variable_id", "table_id"]
].nunique()

,experiment_id,variable_id,table_id
source_id,,,


In [15]:
def drop_all_bounds(ds):
    """Drop coordinates like 'time_bounds' from datasets,
    which can lead to issues when merging."""
    drop_vars = [vname for vname in ds.coords
                 if (('_bounds') in vname ) or ('_bnds') in vname]
    return ds.drop_vars(drop_vars)

def open_dsets(df):
    """Open datasets from cloud storage and return xarray dataset."""
    dsets = [xr.open_zarr(fsspec.get_mapper(ds_url), consolidated=True)
             .pipe(drop_all_bounds)
             for ds_url in df.zstore]
    try:
        ds = xr.merge(dsets, join='exact')
        return ds
    except ValueError:
        return None

def open_delayed(df):
    """A dask.delayed wrapper around `open_dsets`.
    Allows us to open many datasets in parallel."""
    return dask.delayed(open_dsets)(df)

In [18]:
from collections import defaultdict

dsets = defaultdict(dict)
for group, df in col_subset.df.groupby(by=['source_id', 'experiment_id']):
    dsets[group[0]][group[1]] = open_delayed(df)

In [21]:
# %time open_dsets(df)

In [20]:
dsets_ = dask.compute(dict(dsets))[0]

## Data Analysis
- Reduce data via Global Mean
- Grab some observations ?

In [22]:
def get_lat_name(ds):
    """Figure out what is the latitude coordinate for each dataset."""
    for lat_name in ['lat', 'latitude']:
        if lat_name in ds.coords:
            return lat_name
    raise RuntimeError("Couldn't find a latitude coordinate")

def global_mean(ds):
    """Return global mean of a whole dataset."""
    lat = ds[get_lat_name(ds)]
    weight = np.cos(np.deg2rad(lat))
    weight /= weight.mean()
    other_dims = set(ds.dims) - {'time'}
    return (ds * weight).mean(other_dims)

In [23]:
expts = ['piControl', 'abrupt-4xCO2']
expt_da = xr.DataArray(expts, dims='experiment_id',
                       coords={'experiment_id': expts})

dsets_aligned = {}

for k, v in tqdm(dsets_.items()):
    expt_dsets = v.values()
    if any([d is None for d in expt_dsets]):
        print(f"Missing experiment for {k}")
        continue

    for ds in expt_dsets:
        ds.coords['year'] = ds.time.dt.year - ds.time.dt.year[0]

    # workaround for
    # https://github.com/pydata/xarray/issues/2237#issuecomment-620961663
    dsets_ann_mean = [v[expt].pipe(global_mean).swap_dims({'time': 'year'}).drop_vars('time').coarsen(year=12).mean()
                      for expt in expts]

    # align everything with the 4xCO2 experiment
    dsets_aligned[k] = xr.concat(dsets_ann_mean, join='right',dim=expt_da)

0it [00:00, ?it/s]


In [24]:
%%time
dsets_aligned_ = dask.compute(dsets_aligned)[0]

CPU times: user 84 μs, sys: 0 ns, total: 84 μs
Wall time: 81.1 μs


In [1]:
source_ids = list(dsets_aligned_.keys())
source_da = xr.DataArray(source_ids, dims='source_id',coords={'source_id': source_ids})

big_ds = xr.concat([ds.reset_coords(drop=True) for ds in dsets_aligned_.values()],dim=source_da)
big_ds

NameError: name 'dsets_aligned_' is not defined

### Calculated Derived Variables

In [27]:
big_ds['imbalance'] = big_ds['rsdt'] - big_ds['rsut'] - big_ds['rlut']

ds_mean = big_ds[['tas', 'imbalance']].sel(experiment_id='piControl').mean(dim='year')
ds_anom = big_ds[['tas', 'imbalance']] - ds_mean

# add some metadata
ds_anom.tas.attrs['long_name'] = 'Global Mean Surface Temp Anom'
ds_anom.tas.attrs['units'] = 'K'
ds_anom.imbalance.attrs['long_name'] = 'Global Mean Radiative Imbalance'
ds_anom.imbalance.attrs['units'] = 'W m$^{-2}$'

ds_anom

NameError: name 'big_ds' is not defined

In [ ]:
# limit to the gregory 150-year period
first_150_years = slice(0, 149)
ds_anom.tas.sel(year=first_150_years).plot.line(col='source_id', x='year', col_wrap=5)

### Calculate ECS

In [ ]:
ds_abrupt = ds_anom.sel(year=first_150_years, experiment_id='abrupt-4xCO2').reset_coords(drop=True)

In [ ]:
def calc_ecs(ds):
    tas_1d = ds.tas.squeeze()
    imbalance_1d = ds.imbalance.squeeze()

    # Align and drop NaNs
    tas_1d, imbalance_1d = xr.align(tas_1d.dropna('year'), imbalance_1d.dropna('year'), join='inner')

    a, b = np.polyfit(tas_1d.values, imbalance_1d.values, 1)
    ecs = -0.5 * (b / a)
    return xr.DataArray(ecs, attrs={'units': 'K'})

In [ ]:
ds_abrupt['ecs'] = ds_abrupt.groupby('source_id').apply(calc_ecs)
ds_abrupt

In [ ]:
%%time
# Convert to DataFrame
df = ds_abrupt.to_dataframe().reset_index()

# Drop NaNs
df = df.dropna(subset=['tas', 'imbalance'])

# Set up FacetGrid
g = sns.FacetGrid(df, col="source_id", col_wrap=5, height=3.5)
g.map_dataframe(sns.scatterplot, x="tas", y="imbalance")

# Add Gregory fit line and ECS text
def plot_ecs(data, color, **kwargs):
    x = data['tas'].values
    y = data['imbalance'].values
    mask = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[mask], y[mask]
    
    if len(x) < 2:
        return  # skip underpopulated panels
    
    a, b = np.polyfit(x, y, 1)
    ecs = -0.5 * b / a
    
    x_line = np.array([0, x.max()])
    y_line = np.polyval([a, b], x_line)
    
    plt.plot(x_line, y_line, 'k')
    plt.text(0.6 * x.max(), 0.6 * y.max(), f'ECS={ecs:2.2f}K', fontsize=10, weight='bold')
    plt.grid(True)

g.map_dataframe(plot_ecs)

# Optional: adjust layout
g.set_titles("{col_name}")
g.set_axis_labels("Global Mean Tas (K)", "TOA Imbalance (W/m²)")
plt.tight_layout()
plt.show()


In [ ]:
# fg = ds_abrupt.plot.scatter(x='tas', y='imbalance', col='source_id', col_wrap=5)

# def calc_and_plot_ecs(x, y, **kwargs):
#     x = x[~np.isnan(x)]
#     y = y[~np.isnan(y)]
#     a, b = np.polyfit(x, y, 1)
#     ecs = -0.5 * b/a
#     plt.autoscale(False)
#     plt.plot([0, 10], np.polyval([a, b], [0, 10]), 'k')
#     plt.text(4, 6, f'ECS = {ecs:3.2f}', fontdict={'weight': 'bold', 'size': 12})
#     plt.grid()

# fg.map(calc_and_plot_ecs, 'tas', 'imbalance')

In [ ]:
cluster.close()